# CrashLens - AMD GPU Workload Testing
## Real AMD ROCm GPU Testing with Failure Diagnosis

This notebook demonstrates CrashLens running on real AMD GPUs with ROCm.

**Prerequisites:**
- AMD GPU with ROCm installed
- PyTorch with ROCm support
- CrashLens backend running (locally or accessible)

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install requests torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm5.7

## Configuration: CrashLens API Connection

In [ ]:
import requests
import json
import time
import subprocess
import sys
from datetime import datetime

# IMPORTANT: Update this to your CrashLens backend URL
# If running locally with ngrok: use your ngrok URL
# If running on same network: use your local IP
CRASHLENS_API = "https://dominga-floating-shira.ngrok-free.dev"  # UPDATE THIS!

# Headers to bypass ngrok browser warning
HEADERS = {"ngrok-skip-browser-warning": "true"}

print(f"CrashLens API: {CRASHLENS_API}")

# Test connection
try:
    response = requests.get(f"{CRASHLENS_API}/health", headers=HEADERS, timeout=5)
    print("✅ Connected to CrashLens backend!")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"❌ Cannot connect to CrashLens: {e}")
    print("\nOptions:")
    print("1. Update CRASHLENS_API to your backend URL")
    print("2. Use ngrok to expose local backend: ngrok http 8080")
    print("3. Ensure CrashLens is running: docker-compose up -d")

## Verify AMD GPU Availability

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("HIP available:", hasattr(torch.version, 'hip') and torch.version.hip is not None)

if torch.cuda.is_available():
    print(f"\nGPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    
    # Get GPU memory info
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU Memory: {total_memory:.2f} GB")
else:
    print("⚠️ No GPU detected. Running in CPU mode.")

## Check ROCm Version

In [ ]:
# Check rocm-smi availability
try:
    result = subprocess.run(['rocm-smi', '--showproductname'], 
                          capture_output=True, text=True, timeout=5)
    print("ROCm SMI Output:")
    print(result.stdout)
except FileNotFoundError:
    print("rocm-smi not found in PATH")
except Exception as e:
    print(f"Error running rocm-smi: {e}")

## Helper Functions for CrashLens Integration

In [ ]:
class CrashLensWorkload:
    """Helper class to integrate workloads with CrashLens"""
    
    def __init__(self, name, api_url=CRASHLENS_API):
        self.name = name
        self.api_url = api_url
        self.workload_id = None
        self.start_time = None
        self.logs = []
        self.metrics = []
        
    def create(self):
        """Create workload in CrashLens"""
        try:
            response = requests.post(f"{self.api_url}/workloads", 
                                    headers=HEADERS,
                                    json={
                "name": self.name,
                "type": "ML_JOB",
                "status": "running"
            })
            data = response.json()
            self.workload_id = data["id"]
            self.start_time = time.time()
            print(f"✅ Created workload: {self.name} (ID: {self.workload_id})")
            return self.workload_id
        except Exception as e:
            print(f"❌ Failed to create workload: {e}")
            return None
    
    def log(self, message):
        """Add log message"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{timestamp}] {message}"
        self.logs.append(log_entry)
        print(log_entry)
    
    def collect_gpu_metrics(self):
        """Collect GPU metrics using rocm-smi or PyTorch"""
        if not torch.cuda.is_available():
            return
        
        try:
            # PyTorch GPU metrics
            memory_allocated = torch.cuda.memory_allocated(0) / 1e6  # MB
            memory_reserved = torch.cuda.memory_reserved(0) / 1e6
            total_memory = torch.cuda.get_device_properties(0).total_memory / 1e6
            
            metric = {
                "timestamp": datetime.now().isoformat(),
                "gpu_memory_used_mb": int(memory_allocated),
                "gpu_memory_total_mb": int(total_memory),
                "gpu_memory_percent": round((memory_allocated / total_memory) * 100, 2)
            }
            self.metrics.append(metric)
        except Exception as e:
            self.log(f"Failed to collect metrics: {e}")
    
    def update_status(self, status, failure_type=None, exit_code=0):
        """Update workload status in CrashLens"""
        if not self.workload_id:
            return
        
        runtime = time.time() - self.start_time if self.start_time else 0
        
        payload = {
            "status": status,
            "runtime_seconds": runtime,
            "exit_code": exit_code,
            "job_logs": "\n".join(self.logs),
            "gpu_metrics": json.dumps(self.metrics)
        }
        
        if failure_type:
            payload["failure_type"] = failure_type
            payload["wasted_gpu_seconds"] = runtime
        
        try:
            response = requests.put(
                f"{self.api_url}/workloads/{self.workload_id}",
                headers=HEADERS,
                json=payload
            )
            self.log(f"Updated status to: {status}")
        except Exception as e:
            print(f"Failed to update status: {e}")
    
    def diagnose(self):
        """Run AI diagnosis"""
        if not self.workload_id:
            return None
        
        try:
            print(f"\n🤖 Running AI diagnosis...")
            response = requests.post(
                f"{self.api_url}/workloads/{self.workload_id}/diagnose",
                headers=HEADERS
            )
            diagnosis = response.json()
            
            print("\n" + "="*60)
            print("AI DIAGNOSIS REPORT")
            print("="*60)
            print(f"\n📋 Root Cause:\n{diagnosis.get('root_cause', 'N/A')}")
            print(f"\n🔍 Evidence:")
            for evidence in diagnosis.get('evidence', []):
                print(f"  • {evidence}")
            print(f"\n✅ Recommended Fixes:")
            for fix in diagnosis.get('recommended_fixes', []):
                print(f"  • {fix}")
            print(f"\n🛡️ Prevention:\n{diagnosis.get('prevention', 'N/A')}")
            print(f"\n🔄 Safe to Retry: {diagnosis.get('safe_to_retry', 'N/A')}")
            print("="*60)
            
            return diagnosis
        except Exception as e:
            print(f"Failed to get diagnosis: {e}")
            return None

print("✅ CrashLens helper functions loaded")

## Test 1: Successful PyTorch Training

In [ ]:
# Create workload
workload = CrashLensWorkload("PyTorch MNIST Training - Success")
workload.create()

try:
    workload.log("Starting PyTorch training...")
    workload.collect_gpu_metrics()
    
    # Simple training loop
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    workload.log(f"Using device: {device}")
    
    # Create a small model
    model = torch.nn.Sequential(
        torch.nn.Linear(784, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 10)
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters())
    criterion = torch.nn.CrossEntropyLoss()
    
    # Simulate training
    for epoch in range(5):
        # Dummy data
        x = torch.randn(32, 784).to(device)
        y = torch.randint(0, 10, (32,)).to(device)
        
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        workload.log(f"Epoch {epoch+1}/5 - Loss: {loss.item():.4f}")
        workload.collect_gpu_metrics()
    
    workload.log("Training completed successfully!")
    workload.update_status("succeeded")
    
    print("\n✅ Workload succeeded - No diagnosis needed")
    
except Exception as e:
    workload.log(f"ERROR: {str(e)}")
    workload.update_status("failed", failure_type="UNKNOWN_ERROR", exit_code=1)
    workload.diagnose()

## Test 2: GPU Out of Memory (Intentional Failure)

In [ ]:
# Create workload
workload = CrashLensWorkload("PyTorch Training - GPU OOM")
workload.create()

try:
    workload.log("Starting training with large batch size...")
    workload.collect_gpu_metrics()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    workload.log(f"Using device: {device}")
    
    if torch.cuda.is_available():
        # Create a model that will cause OOM
        workload.log("Creating large model...")
        
        # Intentionally create huge tensors to trigger OOM
        huge_tensor = torch.randn(10000, 10000, 100).to(device)
        workload.log(f"Allocated tensor shape: {huge_tensor.shape}")
        workload.collect_gpu_metrics()
        
        # This should fail
        another_huge = torch.randn(10000, 10000, 100).to(device)
    else:
        # Simulate OOM error for CPU mode
        raise RuntimeError("HIP out of memory. Tried to allocate 38.15 GiB")
    
    workload.update_status("succeeded")
    
except RuntimeError as e:
    error_msg = str(e)
    workload.log(f"RuntimeError: {error_msg}")
    workload.collect_gpu_metrics()
    
    # Classify failure
    if "out of memory" in error_msg.lower() or "oom" in error_msg.lower():
        workload.update_status("failed", failure_type="GPU_OUT_OF_MEMORY", exit_code=1)
    else:
        workload.update_status("failed", failure_type="ROCM_ERROR", exit_code=1)
    
    # Get AI diagnosis
    workload.diagnose()

except Exception as e:
    workload.log(f"ERROR: {str(e)}")
    workload.update_status("failed", failure_type="UNKNOWN_ERROR", exit_code=1)
    workload.diagnose()

## Test 3: Dependency Error

In [ ]:
# Create workload
workload = CrashLensWorkload("Training with Missing Dependency")
workload.create()

try:
    workload.log("Importing required libraries...")
    
    # Try to import non-existent library
    import nonexistent_library_xyz123
    
    workload.update_status("succeeded")
    
except ImportError as e:
    workload.log(f"ImportError: {str(e)}")
    workload.update_status("failed", failure_type="DEPENDENCY_ERROR", exit_code=1)
    workload.diagnose()
    
except Exception as e:
    workload.log(f"ERROR: {str(e)}")
    workload.update_status("failed", failure_type="UNKNOWN_ERROR", exit_code=1)
    workload.diagnose()

## Test 4: Missing Checkpoint File

In [ ]:
# Create workload
workload = CrashLensWorkload("Resume Training from Checkpoint")
workload.create()

try:
    workload.log("Loading checkpoint...")
    
    # Try to load non-existent checkpoint
    checkpoint_path = "/path/to/nonexistent/checkpoint.pth"
    workload.log(f"Checkpoint path: {checkpoint_path}")
    
    checkpoint = torch.load(checkpoint_path)
    
    workload.update_status("succeeded")
    
except FileNotFoundError as e:
    workload.log(f"FileNotFoundError: {str(e)}")
    workload.update_status("failed", failure_type="MISSING_CHECKPOINT", exit_code=1)
    workload.diagnose()
    
except Exception as e:
    workload.log(f"ERROR: {str(e)}")
    workload.update_status("failed", failure_type="UNKNOWN_ERROR", exit_code=1)
    workload.diagnose()

## View Results in CrashLens Dashboard

Go to your CrashLens dashboard to see all the workloads:

**Dashboard URL:** http://localhost:3000

You should see:
- ✅ 1 successful workload
- ❌ 3 failed workloads
- 📊 GPU metrics charts
- 🤖 AI diagnosis reports

## Summary

This notebook demonstrates CrashLens running on real AMD GPUs:

1. ✅ **Successful Training** - No diagnosis needed
2. ❌ **GPU OOM** - AI diagnoses memory issues and suggests fixes
3. ❌ **Dependency Error** - Identifies missing packages
4. ❌ **Missing Checkpoint** - Detects file not found errors

All workloads are:
- Tracked in real-time
- GPU metrics collected via PyTorch/rocm-smi
- AI diagnosis provided for failures
- Visible in the CrashLens dashboard

**Next Steps:**
1. Check the dashboard at http://localhost:3000
2. View detailed diagnosis reports
3. Explore GPU metrics charts
4. Use this for your video demo!